In [34]:
import pickle   # importing pickle for saving and loading machine learning models
from sklearn.model_selection import train_test_split  # importing train_test_split for spliting the data into training and testing
from imblearn.over_sampling import SMOTE  # importing SMOTE for Balancing the Data
import warnings
warnings.filterwarnings('ignore')

In [35]:
import pandas as pd
import numpy as np
import seaborn as sns
from ydata_profiling import ProfileReport
from sklearn.preprocessing import StandardScaler,LabelEncoder,OneHotEncoder,OrdinalEncoder
import warnings
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
import pickle
from sklearn.pipeline import Pipeline


class ModifiedLabelEncoder(LabelEncoder):
    def fit_transform(self, y, *args, **kwargs):
        return super().fit_transform(y).reshape(-1, 1)

    def transform(self, y, *args, **kwargs):
        return super().transform(y).reshape(-1, 1)


In [36]:
with open('Processed_data.pkl','rb') as f:
    df = pickle.load(f)

In [37]:
with open('Tree_CT.pkl','rb') as f:
    pre = pickle.load(f)

In [38]:
X = df.drop("Attrition",axis=1)     # Extract the features (all columns except Attritions) from the dataset
y = df["Attrition"].map({"No":0,"Yes":1})  # Extract the target variable from the dataset with converting 0 and 1.

In [39]:
df.Attrition.value_counts()

Attrition
No     1233
Yes     237
Name: count, dtype: int64

In [40]:
# Spliting the data into train and test
x_train,x_test,y_train,y_test = train_test_split(X,y,test_size=0.25,random_state=33)

In [41]:
# Transform the training data using the preprocessor object or PipeLine
processed_x_train = pre.fit_transform(x_train)

In [42]:
## Data Balancing and Creating Model
from imblearn.over_sampling import SMOTE 
from collections import Counter
x_sm,y_sm = SMOTE().fit_resample(processed_x_train,y_train)
print(Counter(y_sm)) 


Counter({1: 927, 0: 927})


In [43]:
## 1️⃣ Import required libraries
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, f1_score

# 2️⃣ Create Decision Tree object (you can add parameters if needed)
dt = DecisionTreeClassifier(random_state=42)  # fixing random_state for reproducibility

# 3️⃣ Train the model on training data
dt.fit(x_sm, y_sm)  # X_sm, y_sm = training data



 

DecisionTreeClassifier(random_state=42)

In [44]:
# 4️⃣ Make predictions
processed_x_test = pre.fit_transform(x_test)
y_train_pred = dt.predict(x_sm)    # predictions on training data
y_test_pred = dt.predict(processed_x_test)   # predictions on testing data


In [45]:

# 5️⃣ Calculate Accuracy
train_acc = accuracy_score(y_sm, y_train_pred)
test_acc = accuracy_score(y_test, y_test_pred)

# 6️⃣ Calculate F1-score
train_f1 = f1_score(y_sm, y_train_pred)
test_f1 = f1_score(y_test, y_test_pred)

# 7️⃣ Print results
print("Training Accuracy:", train_acc)
print("Testing Accuracy:", test_acc)
print("Training F1-score:", train_f1)
print("Testing F1-score:", test_f1)



Training Accuracy: 1.0
Testing Accuracy: 0.7934782608695652
Training F1-score: 1.0
Testing F1-score: 0.45714285714285713


## Hyperparameters of DecisionTree
* Hyperparameter tuning is searching the hyperparameter space for a set of values that will optimize your model architecture.
* Criterion: The function to measure the quality of a split. Supported criteria are "gini" for the Gini impurity and "entropy" for the information gain.


* Splitter: This is how the decision tree searches the features for a split. The default value is set to “best”. That is, for each node, the algorithm considers all the features and chooses the best split. If you decide to set the splitter parameter to “random,” then a random subset of features will be considered.



* max_depth: This determines the maximum depth of the tree.  we use a depth of two to make our decision tree. ... This will often result in over-fitted decision trees. The depth parameter is one of the ways in which we can regularize the tree, or limit the way it grows to prevent over-fitting..The tree perfectly fits the training data and fails to generalize on testing data.



* min_samples_split:Ideal range is 1 to 40.min_samples_split specifies the minimum number of samples required to split an internal node, while min_samples_leaf specifies the minimum number of samples required to be at a leaf node.



* min_samples_leaf: The minimum number of samples required to be at a leaf node.Similarr to min sample split ,this describes the minimum number of samples at the leaf,the base of tree.Ideal range is 1 to 20.(thershold value to make a decision)like 40


# Decision Tree Hyperparameters

| Hyperparameter | Meaning | Small Value Effect | Large Value Effect | Common Options |
|---|---|---|---|---|
| `criterion` | Measures quality of split | — | — | `"gini"`, `"entropy"` |
| `splitter` | Decides how split is selected | — | — | `"best"`, `"random"` |
| `max_depth` | Maximum depth of tree | Underfitting | Overfitting | `1-20` |
| `min_samples_split` | Minimum samples required to split node | Complex Tree | Simpler Tree | `2,3,4...` |
| `min_samples_leaf` | Minimum samples required at leaf node | Detailed Rules | Generalized Rules | `1,2,3...` |

---

# 1. criterion

| Option | Formula | Meaning | Advantage | Disadvantage |
|---|---|---|---|---|
| `gini` | $$Gini = 1 - \sum p_i^2$$ | Measures impurity | Faster | Slightly less informative |
| `entropy` | $$Entropy = -\sum p_i\log_2(p_i)$$ | Measures randomness | Better information gain | Slower |

---

## Example for Gini

Suppose:

| Approved | Rejected |
|---|---|
| 8 | 2 |

Probability:

$$
P(Approved)=\frac{8}{10}=0.8
$$

$$
P(Rejected)=\frac{2}{10}=0.2
$$

Gini:

$$
Gini = 1 - (0.8^2 + 0.2^2)
$$

$$
= 1 - (0.64 + 0.04)
$$

$$
= 0.32
$$

Lower value means:

> Better purity

---

# 2. splitter

| Option | Meaning | Advantage | Disadvantage |
|---|---|---|---|
| `best` | Checks all features and selects best split | Better accuracy | Slower |
| `random` | Randomly selects features | Faster | Slightly lower accuracy |

---

# 3. max_depth

| Depth Value | Result |
|---|---|
| Very Small | Underfitting |
| Balanced | Good Generalization |
| Very Large | Overfitting |

---

## Example

### max_depth = 1

- Very small tree
- Learns fewer patterns

### max_depth = 15

- Very large tree
- May memorize training data

---

# 4. min_samples_split

| Value | Effect |
|---|---|
| Small Value | More splits, complex tree |
| Large Value | Fewer splits, simpler tree |

---

## Example

### min_samples_split = 2

Node can split with only 2 rows.

### min_samples_split = 10

Node needs at least 10 rows to split.

---

# 5. min_samples_leaf

| Value | Effect |
|---|---|
| Small Value | Detailed rules |
| Large Value | Generalized rules |

---

## Example

### min_samples_leaf = 1

Leaf can contain only 1 row.

### min_samples_leaf = 10

Each leaf must contain at least 10 rows.

---

# Why Hyperparameter Tuning?

| Without Tuning | With Tuning |
|---|---|
| May give poor model | Better model |
| Can overfit | Better generalization |
| Default parameters only | Best parameter combination |

---

# Goal of Hyperparameter Tuning

Find balance between:

| Problem | Meaning |
|---|---|
| Underfitting | Model learns too little |
| Overfitting | Model memorizes training data |
| Generalization | Model performs well on new data |

In [46]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, f1_score


# =====================================================
# STEP 1 : Creating Hyperparameter Dictionary
# =====================================================

params = {

    # Method used to measure quality of split
    "criterion": ["gini", "entropy"],

    # Method used to choose split
    "splitter": ["best", "random"],

    # Maximum depth of decision tree
    "max_depth": list(range(1, 20)),

    # Minimum samples needed to split node
    "min_samples_split": [2, 3, 4],

    # Minimum samples needed at leaf node
    "min_samples_leaf": list(range(1, 20))
}


# OUTPUT:
# Dictionary containing all hyperparameter combinations
# GridSearchCV will test all combinations


# =====================================================
# STEP 2 : Creating Decision Tree Model
# =====================================================

dt = DecisionTreeClassifier(random_state=3)


# OUTPUT:
# Empty Decision Tree model object created


# =====================================================
# STEP 3 : Applying GridSearchCV
# =====================================================

grid = GridSearchCV(

    estimator=dt,        # model to train

    param_grid=params,  # hyperparameter combinations

    scoring="f1",       # evaluation metric

    cv=3,               # 3-fold cross validation

    n_jobs=-1,          # use all CPU cores

    verbose=1           # shows training progress
)


# OUTPUT:
# GridSearchCV object created
# Ready for hyperparameter tuning


# =====================================================
# STEP 4 : Training GridSearchCV
# =====================================================

grid.fit(x_sm, y_sm)


# OUTPUT:
# Model gets trained on different parameter combinations
# Best combination selected automatically


# =====================================================
# STEP 5 : Extracting Best Hyperparameters
# =====================================================

best_params = grid.best_params_


# Printing best hyperparameters
print("="*45)
print("      BEST HYPERPARAMETERS")
print("="*45)

for key, value in best_params.items():
    print(f"{key:<20} : {value}")

print("="*45)


# OUTPUT:
# Displays best parameter values selected by GridSearchCV


# =====================================================
# STEP 6 : Creating Final Model Using Best Parameters
# =====================================================

best_dt_model = DecisionTreeClassifier(

    criterion=best_params['criterion'],

    splitter=best_params['splitter'],

    max_depth=best_params['max_depth'],

    min_samples_split=best_params['min_samples_split'],

    min_samples_leaf=best_params['min_samples_leaf'],

    random_state=3
)


# OUTPUT:
# Final optimized Decision Tree model created


# =====================================================
# STEP 7 : Training Final Model
# =====================================================

best_dt_model.fit(x_sm, y_sm)


# OUTPUT:
# Final model trained using best hyperparameters


# =====================================================
# STEP 8 : Making Predictions
# =====================================================

y_train_pred = best_dt_model.predict(x_sm)

y_test_pred = best_dt_model.predict(processed_x_test)


# OUTPUT:
# Predictions generated for training and testing data


# =====================================================
# STEP 9 : Calculating Accuracy
# =====================================================

train_accuracy = accuracy_score(y_sm, y_train_pred)

test_accuracy = accuracy_score(y_test, y_test_pred)


# OUTPUT:
# Accuracy score calculated for train and test data


# =====================================================
# STEP 10 : Calculating F1-Score
# =====================================================

train_f1 = f1_score(y_sm, y_train_pred)

test_f1 = f1_score(y_test, y_test_pred)


# OUTPUT:
# F1-score calculated for train and test data


# =====================================================
# STEP 11 : Printing Final Performance
# =====================================================

print("\n" + "="*45)
print("     DECISION TREE PERFORMANCE")
print("="*45)

print(f"Training Accuracy : {train_accuracy:.4f}")
print(f"Testing Accuracy  : {test_accuracy:.4f}")

print(f"\nTraining F1-Score : {train_f1:.4f}")
print(f"Testing F1-Score  : {test_f1:.4f}")

print("="*45)


# OUTPUT:
# Displays final model performance metrics
# Used to check:
# 1. Model accuracy
# 2. Model generalization
# 3. Overfitting / underfitting

Fitting 3 folds for each of 4332 candidates, totalling 12996 fits
      BEST HYPERPARAMETERS
criterion            : entropy
max_depth            : 19
min_samples_leaf     : 1
min_samples_split    : 2
splitter             : random

     DECISION TREE PERFORMANCE
Training Accuracy : 0.9995
Testing Accuracy  : 0.7418

Training F1-Score : 0.9995
Testing F1-Score  : 0.3165


# Random Forest

In [47]:
# Importing Random Forest Classifier
from sklearn.ensemble import RandomForestClassifier

# Importing GridSearchCV for hyperparameter tuning
from sklearn.model_selection import GridSearchCV


# Creating dictionary of hyperparameters
# Different combinations will be tested
params = {

    # Number of decision trees in forest
    # More trees increase accuracy but also increase training time
    "n_estimators": [50, 100, 150],

    # Maximum depth of each tree
    # Controls overfitting
    "max_depth": [3, 5, 7, 10],

    # Minimum samples required to split internal node
    # Higher value helps reduce overfitting
    "min_samples_split": [2, 5, 10],

    # Minimum samples required at leaf node
    # Prevents very small leaf nodes
    "min_samples_leaf": [1, 2, 4],

    # Number of features considered for best split
    # sqrt -> square root of total features
    # log2 -> log base 2 of total features
    "max_features": ["sqrt", "log2"],

    # Bootstrap sampling
    # True  -> sampling with replacement
    # False -> uses complete dataset
    "bootstrap": [True, False]
}


# Creating Random Forest model object
# random_state ensures same result every execution
rf_clf = RandomForestClassifier(random_state=42)


# Creating GridSearchCV object
rf_cv = GridSearchCV(

    estimator=rf_clf,     # model to train

    param_grid=params,    # hyperparameter combinations

    scoring="f1",         # evaluation metric

    cv=3,                 # 3-fold cross validation

    verbose=1,            # shows training progress

    n_jobs=-1             # uses all CPU cores
)


# Training model on balanced data
rf_cv.fit(x_sm, y_sm)


# Extracting best hyperparameters
best_params = rf_cv.best_params_


# Printing best hyperparameters
print("="*45)
print("   BEST RANDOM FOREST PARAMETERS")
print("="*45)

for key, value in best_params.items():
    print(f"{key:<20} : {value}")

print("="*45)


# Extracting best model from GridSearchCV
best_rf_model = rf_cv.best_estimator_


# Making predictions on training data
y_train_pred = best_rf_model.predict(x_sm)

# Making predictions on testing data
y_test_pred = best_rf_model.predict(processed_x_test)

Fitting 3 folds for each of 432 candidates, totalling 1296 fits
   BEST RANDOM FOREST PARAMETERS
bootstrap            : True
max_depth            : 10
max_features         : log2
min_samples_leaf     : 1
min_samples_split    : 2
n_estimators         : 100


In [48]:
# Importing evaluation metrics
from sklearn.metrics import accuracy_score, f1_score


# ================= TRAINING PERFORMANCE =================

# Calculating training accuracy
train_accuracy = accuracy_score(y_sm, y_train_pred)

# Calculating training F1-score
train_f1 = f1_score(y_sm, y_train_pred)


# ================= TESTING PERFORMANCE =================

# Calculating testing accuracy
test_accuracy = accuracy_score(y_test, y_test_pred)

# Calculating testing F1-score
test_f1 = f1_score(y_test, y_test_pred)


# ================= PRINTING RESULTS =================

print("="*45)
print("        RANDOM FOREST PERFORMANCE")
print("="*45)

print("\nTraining Performance")
print("-"*30)
print(f"Training Accuracy : {train_accuracy:.4f}")
print(f"Training F1-Score : {train_f1:.4f}")

print("\nTesting Performance")
print("-"*30)
print(f"Testing Accuracy  : {test_accuracy:.4f}")
print(f"Testing F1-Score  : {test_f1:.4f}")

print("="*45)

        RANDOM FOREST PERFORMANCE

Training Performance
------------------------------
Training Accuracy : 0.9957
Training F1-Score : 0.9957

Testing Performance
------------------------------
Testing Accuracy  : 0.8668
Testing F1-Score  : 0.5051


# Bagging (Bootstrap Aggregating)

---

# What is Bagging?

Bagging stands for:

$$
\text{Bootstrap Aggregating}
$$

It is an:

- Ensemble Learning Technique

used to:

- Reduce Overfitting
- Reduce Variance
- Improve Model Stability
- Improve Accuracy

---

# Main Idea of Bagging

Instead of training only one model:

- Train multiple models
- Use different random samples of data
- Combine all predictions

Final prediction becomes more stable and accurate.

---

# Block Diagram Flow

| Step | Process | Explanation |
|---|---|---|
| 1 | Original Training Data | Full dataset is given |
| 2 | Bootstrap Sampling | Multiple random samples are created |
| 3 | Train Multiple Models | One model trained on each sample |
| 4 | Predictions | Every model gives prediction |
| 5 | Aggregation | Combine predictions |
| 6 | Final Prediction | Final output generated |

---

# 1. Original Training Data

Suppose dataset contains:

$$
N \text{ samples}
$$

Example:

| Data | Class |
|---|---|
| X1 | 0 |
| X2 | 1 |
| X3 | 0 |
| X4 | 1 |

This is the original dataset.

---

# 2. Bootstrap Sampling

Random samples are created:

- With replacement
- Same size as original dataset

---

# What is "With Replacement"?

A row can be selected multiple times.

Example:

Original Data:

$$
[X1, X2, X3, X4]
$$

Bootstrap Sample:

$$
[X1, X2, X2, X4]
$$

Notice:

$$
X2
$$

appears twice.

---

# Why Bootstrap Sampling?

Each model gets slightly different data.

This creates:

- Diversity among models
- Better generalization

---

# 3. Train Multiple Models

Separate models are trained:

| Bootstrap Sample | Model |
|---|---|
| Sample 1 | Model 1 |
| Sample 2 | Model 2 |
| Sample K | Model K |

Usually Decision Trees are used because:

- They are unstable learners
- Small data changes create different trees

Bagging stabilizes them.

---

# 4. Predictions

Each model predicts independently.

Example:

| Model | Prediction |
|---|---|
| Model 1 | Yes |
| Model 2 | Yes |
| Model 3 | No |
| Model 4 | Yes |

---

# 5. Aggregation

All predictions are combined.

---

# For Classification

Uses:

$$
\text{Majority Voting}
$$

Example:

| Prediction | Count |
|---|---|
| Yes | 3 |
| No | 1 |

Final Prediction:

$$
Yes
$$

---

# For Regression

Uses:

$$
\text{Average of Predictions}
$$

Example:

Predictions:

$$
[100, 120, 110]
$$

Average:

$$
\frac{100+120+110}{3}
$$

$$
=110
$$

Final Prediction:

$$
110
$$

---

# Final Prediction

Combined prediction becomes:

- More stable
- Less noisy
- Better generalized

---


# Advantages of Bagging

| Advantage | Explanation |
|---|---|
| Reduces Overfitting | Multiple models reduce variance |
| Improves Accuracy | Combined prediction is stronger |
| Stable Model | Less sensitive to noise |
| Parallel Training | Models train independently |

---

# Disadvantages of Bagging

| Disadvantage | Explanation |
|---|---|
| Slow Training | Multiple models are trained |
| More Memory Usage | Stores many models |

---


![Bagging Block Diagram](BAGGING.png)

In [49]:
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, f1_score


# =====================================================
# BAGGING
# =====================================================

# Bagging = Bootstrap Aggregating
#
# -> Creates multiple models
# -> Uses random samples of data
# -> Combines all predictions
#
# Goal:
# -> Reduce overfitting
# -> Improve accuracy
# -> Improve stability


# =====================================================
# BASE MODEL
# =====================================================

# Decision Tree used as base model
base_model = DecisionTreeClassifier(random_state=42)


# =====================================================
# BAGGING MODEL
# =====================================================

# Bagging trains multiple Decision Trees
bag_model = BaggingClassifier(

    estimator=base_model,

    random_state=42
)


# =====================================================
# HYPERPARAMETERS
# =====================================================

params = {

    # Number of Decision Trees
    # More trees -> better stability
    # More trees -> slower training
    "n_estimators": [10, 50, 100],


    # Percentage of rows used
    # for each bootstrap sample
    "max_samples": [0.5, 0.7, 1.0],


    # Percentage of features used
    # for training each model
    "max_features": [0.5, 0.7, 1.0],


    # True  -> sampling with replacement
    # False -> sampling without replacement
    "bootstrap": [True, False]
}


# =====================================================
# GRIDSEARCHCV
# =====================================================

# Finds best hyperparameter combination
grid = GridSearchCV(

    estimator=bag_model,

    param_grid=params,

    scoring="f1",

    cv=3,

    n_jobs=-1,

    verbose=1
)


# =====================================================
# TRAINING GRIDSEARCHCV
# =====================================================

grid.fit(x_sm, y_sm)


# =====================================================
# BEST PARAMETERS
# =====================================================

best_params = grid.best_params_

print("="*45)
print("      BEST HYPERPARAMETERS")
print("="*45)

for key, value in best_params.items():
    print(f"{key:<20} : {value}")

print("="*45)


# =====================================================
# FINAL BAGGING MODEL
# =====================================================

best_bag_model = BaggingClassifier(

    estimator=base_model,

    n_estimators=best_params['n_estimators'],

    max_samples=best_params['max_samples'],

    max_features=best_params['max_features'],

    bootstrap=best_params['bootstrap'],

    random_state=42
)


# =====================================================
# TRAINING FINAL MODEL
# =====================================================

best_bag_model.fit(x_sm, y_sm)


# =====================================================
# PREDICTIONS
# =====================================================

y_train_pred = best_bag_model.predict(x_sm)

y_test_pred = best_bag_model.predict(processed_x_test)


# =====================================================
# ACCURACY
# =====================================================

train_accuracy = accuracy_score(y_sm, y_train_pred)

test_accuracy = accuracy_score(y_test, y_test_pred)


# =====================================================
# F1-SCORE
# =====================================================

train_f1 = f1_score(y_sm, y_train_pred)

test_f1 = f1_score(y_test, y_test_pred)


# =====================================================
# FINAL PERFORMANCE
# =====================================================

print("\n" + "="*45)
print("        BAGGING PERFORMANCE")
print("="*45)

print(f"Training Accuracy : {train_accuracy:.4f}")
print(f"Testing Accuracy  : {test_accuracy:.4f}")

print(f"\nTraining F1-Score : {train_f1:.4f}")
print(f"Testing F1-Score  : {test_f1:.4f}")

print("="*45)


# Train High + Test Low -> Overfitting
# Train Low + Test Low  -> Underfitting
# Train Similar + High  -> Good Model

Fitting 3 folds for each of 54 candidates, totalling 162 fits
      BEST HYPERPARAMETERS
bootstrap            : True
max_features         : 0.5
max_samples          : 0.7
n_estimators         : 50

        BAGGING PERFORMANCE
Training Accuracy : 0.9984
Testing Accuracy  : 0.8641

Training F1-Score : 0.9984
Testing F1-Score  : 0.4444


# Boosting

---

# What is Boosting?

Boosting is an:

$$
\text{Ensemble Learning Technique}
$$

where models are trained:

$$
\text{Sequentially}
$$

Each new model tries to:

> Correct the mistakes made by previous models.

---

# Main Goal of Boosting

Boosting mainly tries to reduce:

$$
\text{Bias}
$$

and improve prediction accuracy.

---

# How Boosting Works?

| Step | Process | Explanation |
|---|---|---|
| 1 | Train Weak Learner 1 | First model trained on original data |
| 2 | Calculate Errors | Find wrongly predicted samples |
| 3 | Increase Weights | Give more importance to wrong predictions |
| 4 | Train Weak Learner 2 | Second model focuses more on difficult samples |
| 5 | Repeat Process | Continue for multiple iterations |
| 6 | Combine Models | Combine all weak learners |
| 7 | Final Prediction | Strong final model created |

---



In [50]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, f1_score


# =====================================================
# GRADIENT BOOSTING
# =====================================================

# Gradient Boosting trains models sequentially.
#
# Each new model tries to correct
# errors made by previous model.
#
# Goal:
# -> Reduce bias
# -> Improve accuracy


# =====================================================
# GRADIENT BOOSTING MODEL
# =====================================================

gb_model = GradientBoostingClassifier(

    random_state=42
)


# =====================================================
# HYPERPARAMETERS
# =====================================================

params = {

    # Number of boosting stages
    # More models -> better learning
    # Too many -> overfitting possible
    "n_estimators": [50, 100, 200],


    # Contribution of each model
    # Small value -> slow learning
    # Large value -> fast learning
    "learning_rate": [0.01, 0.05, 0.1, 0.5],


    # Depth of each Decision Tree
    # Small depth -> simple model
    # Large depth -> complex model
    "max_depth": [1, 3, 5],


    # Fraction of samples used
    # Helps reduce overfitting
    "subsample": [0.5, 0.7, 1.0]
}


# =====================================================
# GRIDSEARCHCV
# =====================================================

# Finds best hyperparameter combination

grid = GridSearchCV(

    estimator=gb_model,

    param_grid=params,

    scoring="f1",

    cv=3,

    n_jobs=-1,

    verbose=1
)


# =====================================================
# TRAINING GRIDSEARCHCV
# =====================================================

grid.fit(x_sm, y_sm)


# =====================================================
# BEST PARAMETERS
# =====================================================

best_params = grid.best_params_

print("="*45)
print("      BEST HYPERPARAMETERS")
print("="*45)

for key, value in best_params.items():
    print(f"{key:<20} : {value}")

print("="*45)


# =====================================================
# FINAL GRADIENT BOOSTING MODEL
# =====================================================

best_gb_model = GradientBoostingClassifier(

    n_estimators=best_params['n_estimators'],

    learning_rate=best_params['learning_rate'],

    max_depth=best_params['max_depth'],

    subsample=best_params['subsample'],

    random_state=42
)


# =====================================================
# TRAINING FINAL MODEL
# =====================================================

best_gb_model.fit(x_sm, y_sm)


# =====================================================
# PREDICTIONS
# =====================================================

y_train_pred = best_gb_model.predict(x_sm)

y_test_pred = best_gb_model.predict(processed_x_test)


# =====================================================
# ACCURACY
# =====================================================

train_accuracy = accuracy_score(y_sm, y_train_pred)

test_accuracy = accuracy_score(y_test, y_test_pred)


# =====================================================
# F1-SCORE
# =====================================================

train_f1 = f1_score(y_sm, y_train_pred)

test_f1 = f1_score(y_test, y_test_pred)


# =====================================================
# FINAL PERFORMANCE
# =====================================================

print("\n" + "="*45)
print("    GRADIENT BOOSTING PERFORMANCE")
print("="*45)

print(f"Training Accuracy : {train_accuracy:.4f}")
print(f"Testing Accuracy  : {test_accuracy:.4f}")

print(f"\nTraining F1-Score : {train_f1:.4f}")
print(f"Testing F1-Score  : {test_f1:.4f}")

print("="*45)


# Train High + Test Low -> Overfitting
# Train Low + Test Low  -> Underfitting
# Train Similar + High  -> Good Model

Fitting 3 folds for each of 108 candidates, totalling 324 fits
      BEST HYPERPARAMETERS
learning_rate        : 0.01
max_depth            : 5
n_estimators         : 200
subsample            : 0.5

    GRADIENT BOOSTING PERFORMANCE
Training Accuracy : 0.9617
Testing Accuracy  : 0.8723

Training F1-Score : 0.9608
Testing F1-Score  : 0.5607


In [51]:
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, f1_score


# =====================================================
# XGBOOST
# =====================================================

# XGBoost = Extreme Gradient Boosting
#
# Advanced version of Gradient Boosting.
#
# Models are trained sequentially.
#
# Each new model corrects errors
# made by previous models.
#
# Goal:
# -> Reduce bias
# -> Improve accuracy
# -> Faster and optimized boosting


# =====================================================
# XGBOOST MODEL
# =====================================================

xgb_model = XGBClassifier(

    random_state=42,

    eval_metric='logloss'
)


# =====================================================
# HYPERPARAMETERS
# =====================================================

params = {

    # Number of boosting rounds
    # More models -> better learning
    # Too many -> overfitting possible
    "n_estimators": [50, 100, 200],


    # Learning speed
    # Small value -> slow learning
    # Large value -> fast learning
    "learning_rate": [0.01, 0.05, 0.1, 0.3],


    # Maximum depth of each tree
    # Small depth -> simple model
    # Large depth -> complex model
    "max_depth": [3, 5, 7],


    # Fraction of rows used
    # Helps reduce overfitting
    "subsample": [0.5, 0.7, 1.0],


    # Fraction of columns(features) used
    # Helps create diversity
    "colsample_bytree": [0.5, 0.7, 1.0]
}


# =====================================================
# GRIDSEARCHCV
# =====================================================

# Finds best hyperparameter combination

grid = GridSearchCV(

    estimator=xgb_model,

    param_grid=params,

    scoring="f1",

    cv=3,

    n_jobs=-1,

    verbose=1
)


# =====================================================
# TRAINING GRIDSEARCHCV
# =====================================================

grid.fit(x_sm, y_sm)


# =====================================================
# BEST PARAMETERS
# =====================================================

best_params = grid.best_params_

print("="*45)
print("      BEST HYPERPARAMETERS")
print("="*45)

for key, value in best_params.items():
    print(f"{key:<20} : {value}")

print("="*45)


# =====================================================
# FINAL XGBOOST MODEL
# =====================================================

best_xgb_model = XGBClassifier(

    n_estimators=best_params['n_estimators'],

    learning_rate=best_params['learning_rate'],

    max_depth=best_params['max_depth'],

    subsample=best_params['subsample'],

    colsample_bytree=best_params['colsample_bytree'],

    random_state=42,

    eval_metric='logloss'
)


# =====================================================
# TRAINING FINAL MODEL
# =====================================================

best_xgb_model.fit(x_sm, y_sm)


# =====================================================
# PREDICTIONS
# =====================================================

y_train_pred = best_xgb_model.predict(x_sm)

y_test_pred = best_xgb_model.predict(processed_x_test)


# =====================================================
# ACCURACY
# =====================================================

train_accuracy = accuracy_score(y_sm, y_train_pred)

test_accuracy = accuracy_score(y_test, y_test_pred)


# =====================================================
# F1-SCORE
# =====================================================

train_f1 = f1_score(y_sm, y_train_pred)

test_f1 = f1_score(y_test, y_test_pred)


# =====================================================
# FINAL PERFORMANCE
# =====================================================

print("\n" + "="*45)
print("         XGBOOST PERFORMANCE")
print("="*45)

print(f"Training Accuracy : {train_accuracy:.4f}")
print(f"Testing Accuracy  : {test_accuracy:.4f}")

print(f"\nTraining F1-Score : {train_f1:.4f}")
print(f"Testing F1-Score  : {test_f1:.4f}")

print("="*45)


# Train High + Test Low -> Overfitting
# Train Low + Test Low  -> Underfitting
# Train Similar + High  -> Good Model

Fitting 3 folds for each of 324 candidates, totalling 972 fits
      BEST HYPERPARAMETERS
colsample_bytree     : 0.7
learning_rate        : 0.1
max_depth            : 7
n_estimators         : 50
subsample            : 1.0

         XGBOOST PERFORMANCE
Training Accuracy : 0.9973
Testing Accuracy  : 0.8696

Training F1-Score : 0.9973
Testing F1-Score  : 0.5000
